# Explicit Variable-Step Batched ODE Solver (JAX + Colab GPU)

This notebook provides a batched explicit variable-step RK45 implementation using JAX arrays.
It is designed for Google Colab GPU execution.


In [ ]:
# Colab GPU setup (run once, then Runtime -> Restart runtime)
# If JAX is already GPU-enabled, you can skip this.
%pip -q install -U "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

In [ ]:
import time
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Enable float64 throughout — float32 has only ~7 significant digits,
# which makes tight tolerances (rtol=1e-6, atol=1e-8) unreliable.
jax.config.update("jax_enable_x64", True)

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())
print('Default backend:', jax.default_backend())

In [ ]:
# ---------------------------------------------------------------------------
# Explicit variable-step RK45 (Dormand-Prince) — JAX/GPU batched solver
# ---------------------------------------------------------------------------

def rhs_explicit_single(t, y, lam):
    """Mildly stiff decoupled linear: y' = -lam*y + sin(t).
    Exact solution: y(t) = c*exp(-lam*t) + (-cos(t) + lam*sin(t)) / (lam^2 + 1).
    """
    return -lam * y + jnp.sin(t)


def rk45_step_single(t, y, h, lam):
    """Dormand-Prince 5(4) step; returns (y5, y5 - y4)."""
    k1 = rhs_explicit_single(t,           y,                                   lam)
    k2 = rhs_explicit_single(t + h/5,     y + h*(1/5*k1),                      lam)
    k3 = rhs_explicit_single(t + 3*h/10,  y + h*(3/40*k1 + 9/40*k2),           lam)
    k4 = rhs_explicit_single(t + 4*h/5,   y + h*(44/45*k1 - 56/15*k2 + 32/9*k3), lam)
    k5 = rhs_explicit_single(t + 8*h/9,
                              y + h*(19372/6561*k1 - 25360/2187*k2
                                     + 64448/6561*k3 - 212/729*k4),             lam)
    k6 = rhs_explicit_single(t + h,
                              y + h*(9017/3168*k1 - 355/33*k2 + 46732/5247*k3
                                     + 49/176*k4 - 5103/18656*k5),              lam)
    y5 = y + h*(35/384*k1 + 500/1113*k3 + 125/192*k4
                - 2187/6784*k5 + 11/84*k6)
    k7 = rhs_explicit_single(t + h, y5, lam)  # FSAL: k7 = f(t+h, y5)
    y4 = y + h*(5179/57600*k1 + 7571/16695*k3 + 393/640*k4
                - 92097/339200*k5 + 187/2100*k6 + 1/40*k7)
    return y5, y5 - y4


def rk45_step_batch(t, y, h, lam):
    """Vectorise rk45_step_single across the batch dimension with vmap."""
    return jax.vmap(rk45_step_single)(t, y, h, lam)


def error_norm_batch(y_old, y_new, err, rtol, atol):
    scale = atol + rtol * jnp.maximum(jnp.abs(y_old), jnp.abs(y_new))
    ratio = err / scale
    return jnp.sqrt(jnp.mean(ratio * ratio, axis=1))


def solve_explicit_variable_batch_jax(
    y0_batch,
    lam_batch,
    t_span=(0.0, 6.0),
    rtol=1e-6,
    atol=1e-8,
    h_init=1e-3,
    h_min=1e-7,
    h_max=0.2,
    max_iters=200_000,
    safety=0.9,
    min_factor=0.2,
    max_factor=5.0,
    track_index=0,
):
    """Batched adaptive RK45 in JAX.

    Uses a Python-level adaptive loop (necessary because each trajectory has
    a different number of steps).  vmap parallelises the stage evaluations
    across the batch at each outer iteration.
    """
    t0, tf = float(t_span[0]), float(t_span[1])
    # float64 throughout (jax_enable_x64 must be set before import).
    y   = jnp.array(y0_batch, dtype=jnp.float64)
    lam = jnp.array(lam_batch, dtype=jnp.float64)

    batch = y.shape[0]
    t    = jnp.full((batch,), t0)
    h    = jnp.full((batch,), h_init)
    done = jnp.zeros((batch,), dtype=bool)

    n_accept = jnp.zeros((batch,), dtype=jnp.int32)
    n_reject = jnp.zeros((batch,), dtype=jnp.int32)

    t_hist = [float(t[track_index])]
    y_hist = [np.array(y[track_index])]

    for _ in range(max_iters):
        active = ~done
        if not bool(jnp.any(active)):
            break

        h_eff = jnp.where(active, jnp.minimum(h, tf - t), h)
        h_eff = jnp.clip(h_eff, h_min, h_max)

        y_cand, err_vec = rk45_step_batch(t, y, h_eff, lam)
        err = error_norm_batch(y, y_cand, err_vec, rtol=rtol, atol=atol)

        accept = (err <= 1.0) & active
        reject = (~accept) & active

        t = jnp.where(accept, t + h_eff, t)
        y = jnp.where(accept[:, None], y_cand, y)

        safe_err = jnp.where(err > 0, err, 1e-300)
        factor_accept = jnp.where(
            err == 0.0,
            max_factor,
            jnp.clip(safety * safe_err**(-0.2), min_factor, max_factor),
        )
        factor_reject = jnp.clip(safety * safe_err**(-0.2), min_factor, 1.0)

        h_prop = jnp.where(accept, h_eff * factor_accept, h_eff * factor_reject)
        h = jnp.where(active, jnp.clip(h_prop, h_min, h_max), h)
        done = done | (t >= (tf - 1e-12))

        n_accept = n_accept + accept.astype(jnp.int32)
        n_reject = n_reject + reject.astype(jnp.int32)

        t_hist.append(float(t[track_index]))
        y_hist.append(np.array(y[track_index]))

    return {
        't_final':  np.array(t),
        'y_final':  np.array(y),
        'n_accept': np.array(n_accept),
        'n_reject': np.array(n_reject),
        't_hist':   np.array(t_hist),
        'y_hist':   np.array(y_hist),
    }


def exact_explicit(t, y0, lam):
    """Exact solution for y' = -lam*y + sin(t)."""
    c = y0 - (-1.0 / (lam**2 + 1.0))
    return c * np.exp(-lam * t) + (-np.cos(t) + lam * np.sin(t)) / (lam**2 + 1.0)

In [ ]:
rng = np.random.default_rng(0)
batch_size = 512
dim = 4
t_span = (0.0, 8.0)
tf = t_span[1]

y0_batch  = rng.normal(size=(batch_size, dim))
lam_batch = rng.uniform(0.5, 10.0, size=(batch_size, dim))

cfg = dict(t_span=t_span, rtol=1e-6, atol=1e-8, h_init=1e-3, h_max=0.1, track_index=0)

# --- Warmup: first call triggers JAX tracing / XLA compilation ---
print("Warming up JAX (first call includes JIT compilation)...")
_ = solve_explicit_variable_batch_jax(
    y0_batch[:4], lam_batch[:4], max_iters=10, **cfg,
)

# --- Timed run ---
start = time.perf_counter()
out = solve_explicit_variable_batch_jax(y0_batch, lam_batch, **cfg)
elapsed = time.perf_counter() - start

print(f"\nExplicit variable-step RK45 (JAX/GPU)  batch={batch_size}  dim={dim}")
print(f"  Runtime       : {elapsed:.3f} s")
print(f"  Throughput    : {batch_size/elapsed:.0f} trajectories/s")
print(f"  Mean accepted : {out['n_accept'].mean():.1f} steps")
print(f"  Mean rejected : {out['n_reject'].mean():.1f} steps")

# --- Correctness vs exact solution ---
errors = []
for i in range(batch_size):
    y_exact = exact_explicit(tf, y0_batch[i], lam_batch[i])
    errors.append(np.max(np.abs(out['y_final'][i] - y_exact)))
print(f"\n  Max |error| vs exact : {np.max(errors):.2e}  (rtol=1e-6, atol=1e-8)")
print(f"  Mean|error| vs exact : {np.mean(errors):.2e}")

# --- Plot tracked trajectory ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(out['t_hist'], out['y_hist'][:, 0])
axes[0].set_xlabel('t'); axes[0].set_ylabel('y[0]')
axes[0].set_title('Tracked trajectory (explicit RK45, JAX)')
axes[0].grid(True)

axes[1].semilogy(sorted(errors))
axes[1].axhline(1e-6, ls='--', color='r', label='rtol')
axes[1].set_xlabel('Trajectory index (sorted)')
axes[1].set_ylabel('|error| at t=tf')
axes[1].set_title('Per-trajectory error vs exact')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()